In [0]:
dbutils.library.restartPython()

In [0]:
from common_utils.bronze import read_raw, bronze_ingestor
from common_utils.logging import get_logger
from datetime import date

import json

# ============================================================
# 0. LOGGER
# ============================================================
logger = get_logger("ds2b")

# ============================================================
# 1. CONFIG PATH
# ============================================================

dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

# ============================================================
# 2. READ CONFIG
# ============================================================
with open(config_path, "r") as f:
    config = json.load(f)

# ============================================================
# 3. RUN DATE
# ============================================================
run_date = date.today().isoformat()
sources = config["sources"]

# ============================================================
# 4. BASE RAW PATH
# ============================================================
base_raw_path = "/Volumes/employeedataplatform/bronze/raw_data/"

for source_name, config in sources.items():

    # --------------------------------------------------------
    # Build raw path
    # --------------------------------------------------------
    raw_path = (
                f"{base_raw_path}/"
                f"load_date={run_date}/"
                f"{config['raw_folder']}/"
                f"{source_name}.{config["file_format"]}"
    )

    logger.info(
        "[%s]starting bronze ingestion",
        source_name
    )

    logger.info(
        "[%s] reading from path %s", 
        source_name, 
        raw_path
    )
    # --------------------------------------------------------
    # Read Raw
    # --------------------------------------------------------
    df = read_raw(
        raw_path,
        file_format=config['file_format'],
        options=config['read_options']
    )
    # --------------------------------------------------------
    # Bronze ingestion
    # --------------------------------------------------------
    rows = bronze_ingestor(
        df, 
        mode=config['mode'], 
        target_table=config['target_table']
    )

    logger.info(
        "[%s] ingested %s rows into %s",
        source_name,
        rows,
        config['target_table']
    )
# ============================================================
# 6. COMPLETION
# ============================================================
logger.info(
    "bronze ingestion completed for all sources"
)